# Section J – Short-answer questions

Answers to J1–J4 (max 80 words each).

## J1 – EXPLAIN FORMATTED

**Query:** `EXPLAIN FORMATTED SELECT dp.category_l1, SUM(fol.net_amount) FROM gold.fact_order_line fol JOIN gold.dim_product dp ON fol.product_id = dp.product_id GROUP BY dp.category_l1`

**Physical join operator:** `PhotonBroadcastHashJoin (Inner)` — the engine chose it because `dim_product` is small (60 rows), so broadcasting avoids shuffling the larger `fact_order_line`.

**Shuffle operator:** `PhotonShuffleExchangeSink` (nodes 3 and 9) — node 3 shuffles `dim_product` to `SinglePartition` for broadcast; node 9 shuffles partial aggregates via `hashpartitioning(category_l1, 16)`.

## J2 – DROP TABLE on managed vs external

When you run `DROP TABLE` on a **managed** table, both the table metadata in the metastore and the underlying data files in cloud storage are permanently deleted.

If it were an **external** table, `DROP TABLE` would only remove the metadata from the metastore; the data files in cloud storage would remain intact and accessible.

## J3 – VACUUM and time travel

No, the F1 time-travel query would fail. VACUUM with default settings removes data files older than 7 days. Eight days after Run 2, the version-0 files (before Batch 2) are older than 7 days and would be deleted, making `VERSION AS OF 0` impossible. The table property that controls how long old data files are kept is **`delta.deletedFileRetentionDuration`**.

## J4 – Promote job from Dev to Prod with DABs

The configuration file is **`databricks.yml`**.

* **`resources.jobs`** — defines the job (tasks, clusters, schedule).
* **`variables`** — defines a `catalog` variable with a dev default (e.g. `fk_ce_2026_07_1378`).
* **`targets`** — defines `dev` and `prod` environments, each overriding the `catalog` variable (e.g. `fk_prod` for prod).